# Text atlas

Three executable stages share text_core.py with the Streamlit app.
Set PROJECT_ROOT to this project's directory before running from another location.

Adapted from INRIA scikit-learn MOOC contributors, CC BY 4.0:
https://github.com/INRIA/scikit-learn-mooc/blob/3d1e8cdf7df6675d8a47d352d66b29dfea36587c/notebooks/dimred_text.ipynb

Corpus: 1,250 historical Wikinews articles, Wikinews contributors,
curated by Mega Rhyme and subsampled by INRIA; separately licensed CC BY 2.5.
See LICENSE, DATA_LICENSE, DATA_SOURCES.md, and NOTICE.

English stop-word TF-IDF → centered randomized PCA (at most 50 components)
→ KMeans in retained PCA space. The map uses two coordinates; both variance
metrics refer to original TF-IDF variance. Categories never enter fitting.
Cluster words use mean original TF-IDF weights. Silhouette is an internal
diagnostic; clusters are not validated topics. The 2D view drops detail.


In [ ]:
# Stage 1 — Load the unchanged corpus
import json
import os
import sys
from pathlib import Path

project_root = Path(globals().get("PROJECT_ROOT", os.environ.get("PROJECT_ROOT", "."))).resolve()
if not (project_root / "text_core.py").is_file():
    raise FileNotFoundError("Supply PROJECT_ROOT pointing to the Text atlas project.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from text_core import load_corpus, analyze

corpus = load_corpus()
print(f"Loaded {len(corpus):,} historical Wikinews articles in original CSV order.")


In [ ]:
# Stage 2 — Run the same analysis as the app
analysis = analyze(min_df=5, max_df=0.8, n_clusters=5, seed=42)
print(f"Retained {analysis['parameters']['n_components']} centered PCA components.")


In [ ]:
# Stage 3 — Report metrics and overwrite results.json in the execution directory
metrics = {
    "articles": len(corpus),
    "vocabulary_size": len(analysis["vocabulary"]),
    "silhouette": analysis["silhouette"],
    "displayed_variance": analysis["displayed_variance"],
    "retained_variance": analysis["retained_variance"],
}
results = {
    **metrics,
    "parameters": analysis["parameters"],
    "coordinates": analysis["coordinates"].tolist(),
    "labels": analysis["labels"].tolist(),
    "vocabulary": analysis["vocabulary"],
    "top_terms": analysis["top_terms"],
    "cluster_document_counts": {
        cluster: int((analysis["labels"] == int(cluster)).sum())
        for cluster in analysis["top_terms"]
    },
}
Path("results.json").write_text(
    json.dumps({"results": results}, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(json.dumps(metrics, indent=2, allow_nan=False))
print("Wrote fresh results.json in the current execution directory.")
